# 29 — AI System Engineering Capstone

## Final Principle
Do not ask how to make a prompt sound better. Ask what behavior is needed, how it is measured, what caused a failure, and what smallest system change improves it safely.

## The Capstone Readiness Gate
This notebook provides a strict `pydantic` validator for an Enterprise AI project. Before a system is deployed to production, it must successfully pass this checklist, providing explicit documentation and evidence for all 8 required deliverables.

In [ ]:
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional


## Step 1: Define the Deliverable Schemas

We model each of the 8 required components of an Enterprise AI System as a Pydantic class.

In [ ]:
class BusinessOutcome(BaseModel):
    goal: str = Field(..., min_length=10)
    risk_boundary: str = Field(..., description="What the model is explicitly NOT allowed to do.")

class Contracts(BaseModel):
    input_schema_defined: bool
    output_schema_defined: bool
    
    @field_validator('input_schema_defined', 'output_schema_defined')
    def must_be_true(cls, v):
        if not v:
            raise ValueError("Contracts MUST be defined.")
        return v

class ArchitectureDecisionRecord(BaseModel):
    selected_pattern: str = Field(..., description="e.g., Simple Prompt, RAG, Agent")
    reason_simpler_patterns_rejected: str = Field(..., min_length=20)

class ValidationSuite(BaseModel):
    regression_tests_passing: bool
    adversarial_tests_passing: bool
    baseline_metric: float = Field(..., description="Previous success rate")
    new_metric: float = Field(..., description="New success rate")

    @field_validator('new_metric')
    def must_improve(cls, v, info):
        if 'baseline_metric' in info.data and v <= info.data['baseline_metric']:
            raise ValueError("New metric MUST improve upon the baseline.")
        return v

class ObservabilityAndRelease(BaseModel):
    telemetry_enabled: bool
    human_in_the_loop_gate: bool
    rollback_plan_defined: bool


## Step 2: The Capstone Project Root

The root project schema aggregates all deliverables.

In [ ]:
class CapstoneProject(BaseModel):
    project_name: str
    outcome: BusinessOutcome
    contracts: Contracts
    architecture: ArchitectureDecisionRecord
    validation: ValidationSuite
    release_ops: ObservabilityAndRelease
    
    def print_readiness_report(self):
        print(f"✅ READY FOR PRODUCTION: {self.project_name}")
        print("\n--- Enterprise Checklist Passed ---")
        print(f"1. Business Outcome: {self.outcome.goal}")
        print(f"   Risk Boundary: {self.outcome.risk_boundary}")
        print(f"2. Contracts: Strictly Enforced I/O")
        print(f"3. ADR: Selected '{self.architecture.selected_pattern}'.")
        print(f"4. Validation: Improved from {self.validation.baseline_metric} to {self.validation.new_metric}")
        print(f"5. Release Ops: Telemetry & Rollbacks active.")


## Step 3: Evaluating a Project

Let's test an incomplete project, and then a fully compliant project.

In [ ]:
print("=== Attempting to Release an Incomplete Project ===")
try:
    bad_project = CapstoneProject(
        project_name="Customer Support Chatbot v1",
        outcome=BusinessOutcome(goal="Answer questions", risk_boundary="Don't be mean"),
        contracts=Contracts(input_schema_defined=True, output_schema_defined=False), # FAIL
        architecture=ArchitectureDecisionRecord(selected_pattern="Agent", reason_simpler_patterns_rejected="Agents are cool"),
        validation=ValidationSuite(regression_tests_passing=True, adversarial_tests_passing=True, baseline_metric=0.8, new_metric=0.7), # FAIL
        release_ops=ObservabilityAndRelease(telemetry_enabled=False, human_in_the_loop_gate=False, rollback_plan_defined=False) # FAIL
    )
except Exception as e:
    print("🚨 PRODUCTION GATE REJECTED:\n", e)


In [ ]:
print("\n=== Releasing a Fully Compliant Enterprise Project ===")
good_project = CapstoneProject(
    project_name="Customer Support Support Copilot v2",
    outcome=BusinessOutcome(
        goal="Draft technical responses for human review, reducing handle time by 15%.", 
        risk_boundary="Must not directly email customers or modify billing data."
    ),
    contracts=Contracts(input_schema_defined=True, output_schema_defined=True),
    architecture=ArchitectureDecisionRecord(
        selected_pattern="RAG + Tool Use", 
        reason_simpler_patterns_rejected="Zero-shot lacked knowledge of private docs. Full agents hallucinated too often."
    ),
    validation=ValidationSuite(
        regression_tests_passing=True, 
        adversarial_tests_passing=True, 
        baseline_metric=0.80, 
        new_metric=0.92
    ),
    release_ops=ObservabilityAndRelease(
        telemetry_enabled=True, 
        human_in_the_loop_gate=True, 
        rollback_plan_defined=True
    )
)

good_project.print_readiness_report()


## Conclusion

Prompt Engineering in the Enterprise is indistinguishable from rigorous Software Engineering.

Congratulations on completing the curriculum.